In [ ]:
# >> FILE OVERVIEW: interactive single-region inference demo (upstream AMW).
# >> Runs the same engine as production (inference_engine.InferenceEngine) on one small
# >> region, so you can look at results on a map. Production runs use
# >> scripts/pipeline.py -> inference_pipeline.py, with DIFFERENT settings (below).
# >> Must be run from core/ so `import gee` etc. resolve as local modules.
from datetime import datetime
import logging
import os
from pathlib import Path
import sys

import geopandas as gpd

# >> gee = Earth Engine download; inference_engine = CNN scoring; tile_utils = tiling.
import gee
from inference_engine import InferenceConfig, InferenceEngine, MaskConfig
import tile_utils

%load_ext autoreload
%autoreload 2

In [ ]:
# >> Region to score: a GeoJSON polygon file under data/boundaries/.
region_name = 'test_region_small'
region_path = f'../data/boundaries/{region_name}.geojson'

# >> GOTCHA: this is an OLDER model name (SSL4EO-MLP ensemble), not the production CNN
# >> 48px_v4.10b-18d-20g-21a-22bc-ensemble. An MLP that expects foundation-model
# >> embeddings will not match embedding_strategy="none" (raw 48x48x13 pixels), which
# >> is the default below; swap in the production .h5 to reproduce production.
model_name = '48px_v0.X-SSL4EO-MLPensemble_2025-10-21'
model_path = f'../models/{model_name}.h5'

# >> Composite date window. Earth Engine filterDate END IS EXCLUSIVE, so 2024-12-31
# >> itself is left out (use 2025-01-01 to include it).
start_date='2024-01-01'
end_date='2024-12-31'
    
# >> Download settings. DIFFERS FROM PRODUCTION: tilesize 576 with pad 0 here, versus
# >> production 552 + 12 pad per side (also 576 px, but tiles do not overlap here).
data_config = gee.DataConfig(
    tilesize=576,
    pad=0, 
    # >> S2L1C = Sentinel-2 Level-1C (top-of-atmosphere), 13 bands, via Earth Engine.
    collection="S2L1C",
    # >> Cloud Score+ cs_cdf >= 0.6 keeps a pixel in a scene (0 = cloudy, 1 = clear).
    clear_threshold=0.6,
    # >> Parallel download threads (production default is 8).
    max_workers=4, # Turn this down from 8 if cacheing, to ease memory pressure 
    # >> None = no on-disk pixel cache; every run re-downloads from Earth Engine.
    image_cache_dir=None,
)

# >> Scoring settings.
inference_config = InferenceConfig(
    model_path=model_path,
    # >> Keep only chips with ensemble mean score > 0.85 (production raw floor is 0.4).
    pred_threshold=0.85,
    # >> stride = chip_size // stride_ratio = 48 // 2 = 24 px (chips overlap by half).
    stride_ratio=2, 
    # >> Chip size in pixels: 48 px at 10 m = 480 m (about 23 ha).
    geo_chip_size=48,
    # >> No embedding cache: embedding_strategy stays at its default "none" (raw pixels).
    embeddings_cache_dir=None,
    # >> SAM2 mask refinement switched off for this demo.
    run_sam2=False
)

# >> Default SAM2 mask settings (unused while run_sam2=False).
mask_config = MaskConfig()


In [ ]:
# >> Load the region polygon and cover it with DLTile squares (descarteslabs grid maths
# >> only, no API calls). Tile key format tilesize:pad:resolution:UTMzone:x:y.
region = gpd.read_file(region_path).geometry.__geo_interface__
tiles = tile_utils.create_tiles(region, data_config.tilesize, data_config.pad)
print(f"Created {len(tiles):,} tiles")

# >> Send the engine's log messages to the notebook output.
logger = logging.getLogger()
stream_handler = logging.StreamHandler(sys.stdout)
logger.addHandler(stream_handler)

# >> Build the engine: it owns the Earth Engine extractor and loads the Keras model.
engine = InferenceEngine(
        start_date=start_date,
        end_date=end_date,
        data_config=data_config,
        config=inference_config,
        mask_config=mask_config,
        logger=logger,
    )

In [ ]:
# Single tile inference for debugging
# >> Download + chip + score ONE tile, no threads. Returns a GeoDataFrame of chips
# >> above pred_threshold with a `confidence` column (mean of the ensemble members).
preds_gdf = engine.predict_on_tile(tiles[0])
display(preds_gdf)

In [ ]:
# >> Full run: max_workers (4) producer threads download tiles into a queue, one consumer
# >> runs the model. Writes GeoJSON to
# >> data/outputs/<model version>/<region>_<version>_0.85_<start>_<end>.geojson
# >> (version = first two "_" tokens of the model file name).
preds = engine.bulk_predict(tiles, Path(region_path).stem)

In [ ]:
# >> Interactive web map (folium) of detections; > 0.85 is a no-op filter here because
# >> the engine already dropped everything at or below pred_threshold.
preds[preds.confidence > 0.85].explore()